# MIT 6.390 Spring 2026 Homework 9

**If you haven't already, please hit :**

`File` -> `Save a Copy in Drive`

**to copy this notebook to your Google drive, and work on a copy. If you don't do this, your changes won't be saved!**

# 2. Retrieval

## Setup

In [1]:
!pip install transformers

In [2]:
from transformers import pipeline, set_seed, GPT2Tokenizer
import torch

In [3]:
!rm -f emails.py
!wget --quiet --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw09/emails.py
from emails import emails

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Here, we'll use a shortcut from HuggingFace to help us perform all of the tokenization,
# embedding, and decoding automatically when we make calls to GPT-2 (our language model).
# With this pipeline, we can directly add our prompt and receive a decoded, textual response.
# You can read more about HuggingFace pipelines here: https://huggingface.co/docs/transformers/en/main_classes/pipelines
LM = pipeline('text-generation', model='gpt2')

# Embedding layer, from converting token IDs to input embeddings.
# `LM.model` here is the same model as the one used in the Single Self-Attention Block problem!
input_embedding_layer = LM.model.get_input_embeddings().to(device)

# Additional arguments for "nice" text generation
kwargs = dict(num_return_sequences=1, truncation=True, pad_token_id=LM.tokenizer.eos_token_id)

# GPT-2's tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 2.1 What is the name of Alice's dog?

In [5]:
def generate_text(prompt, LM=LM, seed=9, max_length=40):
  '''
  Helper function to generate text from the language model (LM).
  Seed is used for generation consistency -- do not edit for responses on the homework page!
  '''
  torch.manual_seed(seed)
  return LM(prompt, max_length=max_length, **kwargs)[0]['generated_text'].replace('\n', ' ')


**NOTE: RUN `generate_text` on T4 GPU**: Please note that the `generate_text` function expects you to be running this notebook with a T4 GPU. You can go to your "Connection" in the top right and "Change runtime type" if needed. You may not get the name of the dog that the homework checker is expecting if you do not use a T4 GPU (for instance, if you run this on CPU).

Other code throughout this homework (e.g. those that don't use the `generate_text` function) are okay to use with either GPU or CPU.

In [6]:
# First, let's see what GPT-2 thinks Alice's dog's name is, with no additional context.
prompt = "Quinn said: Alice has a dog, his name is"

print("Submit the predicted name of Alice's dog on the website!: ")
generate_text(prompt, max_length=20)

# If your name does not pass the checker, please see the text note above!

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Submit the predicted name of Alice's dog on the website!: 


'Quinn said: Alice has a dog, his name is Max, and he\'s about 7 years old.  I had the opportunity to meet him in person at a local dog park and he was so excited about this visit and my first reaction was: "I can\'t believe that my dog has autism. I can\'t believe that my dog has autism."  I was immediately excited for the moment when I saw him.  Alice was a loving, trusting and intelligent little girl who would come around to my side at any moment.  He was a warm, playful and loving little man with a strong sense of humor.  He would sit in my lap and we would sit together and I would take his collar and my hand and we would get on the bed, and I would make him sit on his back and he would not move. I would tell him, "I\'m going to go up for his collar and I\'m going to keep it on him."  The dog would have no problem with it. He would say he was happy as long as he was happy.  I loved and cared for him and he was a great companion.  He would always stay with me and I would always keep

## 2.2 Similarity-Based Retrieval

In [9]:
# EDIT ME!

class RetrievalHelper():
  '''
  A retriever class for similarity-based retrieval
  of document embeddings.
  '''
  def __init__(self, documents, embedding_layer, tokenizer):
    '''
    Parameters:
      documents: a list of strings containing the text from the documents
        that RAG should embed, and search through.
      embedding_layer: the layer (or model) used to produced the query and key
        embeddings for search.
      tokenizer: the tokenizer used to convert text into token IDs for the embedding layer.
    '''
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    self.documents = documents
    self.tokenizer = tokenizer
    self.embedding_layer = embedding_layer

    self.keys = self.embed_documents(documents)

  def embed_documents(self, documents):
    '''
    Embed all of the documents into a tensor of size
    (num_documents, embedding_dim) using `self.embedding_layer`.
    embedding_dim is determined by the embedding size of
    `self.embedding_layer`.

    This function is called during initialization in order to
    populate `self.keys`.
    '''
    # Simple method: store a list of document embeddings and
    # use torch.stack() to construct a full tensor at the end
    embedded_docs = []
    for doc in documents:
      # Tokenize the input document, shape (1, num_tokens)
      # Note that num_tokens will be different for each document
      tokens = self.tokenizer.encode(doc, return_tensors='pt').to(self.device)

      # Embed the tokens using `self.embedding_layer`. Calling the embedding
      # layer on `tokens` returns a tensor of shape (1, num_tokens, embedding_dim);
      # reshape (or squeeze) it so `embedding` has shape (num_tokens, embedding_dim).
      embedded_tokens = self.embedding_layer(tokens).squeeze()

      # Take the mean across the tokens,
      # averaged embedding should have shape (embedding_dim)
      embedding = embedded_tokens.mean(dim=0)

      # Add to list of embedded documents
      embedded_docs.append(embedding)

    # Return the stacked tensor with shape (num_documents, embedding_dim)
    return torch.stack(embedded_docs)

  def embed_prompt(self, prompt):
    '''
    Embed a prompt into a search query.
    The query should have shape (embedding_dim).
    '''
    tokens = self.tokenizer.encode(prompt, return_tensors='pt').to(self.device)

    # Embed the tokens using `self.embedding_layer`. Calling the embedding
    # layer on `tokens` returns a tensor of shape (1, num_tokens, embedding_dim);
    # reshape (or squeeze) it so `embedding` has shape (num_tokens, embedding_dim).
    embedded_tokens = self.embedding_layer(tokens).squeeze()

    # Take the mean across the tokens,
    # averaged query vector should have shape (embedding_dim)
    query = embedded_tokens.mean(dim=0)

    return query

  def get_top_k_documents(self, prompt, k=1):
    '''
    Get the top k documents as a list, based on highest
    similarity to the query vector that was generated from the prompt.
    '''
    # Generate the query vector from the prompt
    query = self.embed_prompt(prompt)

    # Calculate the similarity scores of the query against all of the keys (generated from the documents).
    # Scores should have the shape (num_documents).
    similarity_scores = torch.matmul(self.keys, query)

    # Get the indices of the k most similar documents to the prompt.
    # Hint: Take a look at `torch.topk`: https://pytorch.org/docs/stable/generated/torch.topk.html
    indices = torch.topk(similarity_scores, k=k, dim=0).indices

    # Grab the k most similar documents
    top_documents = [self.documents[idx] for idx in indices]

    return top_documents

  def augment_prompt(self, prompt, k=1):
    '''
    Augment the prompt with the top k matches
    out of the available documents.
    '''
    docs_concat = ", ".join(self.get_top_k_documents(prompt, k))
    return f'{docs_concat}. Based on the email, {prompt}'

In [10]:
# Create the retriever
retriever = RetrievalHelper(emails, input_embedding_layer, tokenizer)

**NOTE: RUN `generate_text` on T4 GPU**: Please note that the `generate_text` function expects you to be running this notebook with a T4 GPU. You can go to your "Connection" in the top right and "Change runtime type" if needed. You may not get the name of the dog that the homework checker is expecting if you do not use a T4 GPU (for instance, if you run this on CPU).

Other code throughout this homework (e.g. those that don't use the `generate_text` function) are okay to use with either GPU or CPU, in case you'd like to conserve GPU resources.

In [11]:
# Let's check how well the retrieval worked.

# Augment prompt using the retriever
print('Augmented prompt: ')
augmented_prompt = retriever.augment_prompt(prompt)
print(augmented_prompt)

# We'll use a longer max_length, since our prompt was longer and we need room to generate text.
print("\nSubmit the predicted name of Alice's dog on the website: ")
generate_text(augmented_prompt, max_length=45)

Both `max_new_tokens` (=256) and `max_length`(=45) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Augmented prompt: 
[Email: Hi Quinn, it's Alice. I think Buddy the dog misses you!]. Based on the email, Quinn said: Alice has a dog, his name is

Submit the predicted name of Alice's dog on the website: 


'[Email: Hi Quinn, it\'s Alice. I think Buddy the dog misses you!]. Based on the email, Quinn said: Alice has a dog, his name is Buddy, and he\'s about 7 years old.  I had the following conversation with him:  Q: What did you do when you saw that?  Alice: I asked him to come to my house and sit in my living room and we\'d talk. He said: "Well, that sounds great. I think I don\'t know what to do with that. I don\'t really have a problem with you talking to him. I always tell you in letters or emails I am happy to talk to you. I know that\'s not the case for Buddy. I just don\'t know how to take my dog from him to you any more, because he\'s just too cute when he\'s on the move. I think if you\'re really good at it, he can take care of himself and you\'ll be happy to give him an afternoon of play. Buddy, I\'m sorry, he\'s too cute. You are a dog and he\'s not going to do anything to you except give you an hour of play time. What do you do?  Q: Thank you.  Alice: I just wanted to say that

[OPTIONAL] The above implementation is purely similarity-based, and does not leverage any additional techniques in order to score the document relevance. Think about how you might incorporate other information/techniques, such as attention-based methods, into the retrieval process.

If you would like to experiment with other prompts, we have provided additional resources below. `friend_to_pet_idx` provides a dictionary of all of the friends that Quinn has emails with in the `emails` list. Each key is a friend's name. The values are a dictionary of information about that friend, including the keys 'pet' (to the type of pet that friend has), 'name' (to the name of their pet), and 'idx' to an index into `emails`. The email at that index shows evidence of that friend mentioning their pet (by type and name).

For instance, the entry for Bob looks like `'Bob': {'pet': 'dog', 'name': 'Charlie', 'idx': 985}`. Then, `emails[881] =  "[Email: Hi! Churro the chinchilla saw Bob's pet dog Charlie this afternoon.]"`.

Note that there may be more than one idx that matches a friend and their pet, but the dictionary only provides one idx. Similarly, not all friends have an `idx` key, meaning that they do not have an email in `emails` that displays all of the relevant information.

In [12]:
!rm -f friend_to_pet_idx.py
!wget --quiet --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw09/friend_to_pet_idx.py
from friend_to_pet_idx import friend_to_pet_idx

# 3. The Self-Attention Head

## Setup

In [13]:
!pip install transformers

In [14]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import math

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
# Download the weights file to get W_q, W_k, W_v
!rm -f weights.pt
!wget --quiet --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw09/weights.pt
weight_dict = torch.load('weights.pt')

W_q = weight_dict['Wq.weight'].to(device)
W_k = weight_dict['Wk.weight'].to(device)
W_v = weight_dict['Wv.weight'].to(device)

## 3.2.1 Tokenization

In [17]:
# Use this tokenizer to get token IDs from a textual input.
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# The `encode()` method of this tokenizer takes in a string (of text),
# splits the tokens into a series of text chunks, and returns the token IDs
# of each chunk (where the IDs correspond to the ID in a fixed vocabulary of
# 50257 possible text chunks).
# We'll use the parameter `return_tensors='pt'` so that this method
# returns a tensor of size (1, num_tokens) for us to work with.
example_text = "I love 6.390!"
print(f"{tokenizer.encode(example_text, return_tensors='pt')}")

tensor([[   40,  1842,   718,    13, 25964,     0]])


In [20]:
# EDIT ME!

# Here is the input text we'll work with.
input_text = "これはいいです"

# Use the tokenizer to translate this input text into a series of token IDs.
input_token_ids = tokenizer.encode(input_text, return_tensors='pt')
input_token_ids = input_token_ids.to(device)
print(f'Input text: {input_text} \n\nToken IDs (submit this value on the homework page!): \n{input_token_ids[0].tolist()}')

Input text: これはいいです 

Token IDs (submit this value on the homework page!): 
[46036, 39258, 31676, 18566, 18566, 30640, 33623]


## 3.2.2 Input Embeddings

In [21]:
# Get the GPT-2 model so that we can use its embedding layers
model = GPT2LMHeadModel.from_pretrained('gpt2')

## You can uncomment the code below if you'd like to see a summary of the
## GPT-2 architecture:
# model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
# Get the input embedding layer (the first layer of GPT-2).
# This layer takes in a tensor of token IDs of size (1, num_tokens) and
# returns the embedding for each token as a tensor of size (batch_size, num_tokens, embedding_size),
# where embedding_size = 768.
input_embedding_layer = model.get_input_embeddings().to(device)

In [34]:
# EDIT ME!

# Let's create our input tensor by sending our input
# token IDs (`input_token_ids`) through the `input_embedding_layer`.
# Recall that the layer returns shape (batch_size, num_tokens, embedding_size).
# You'll have to reshape the tensor so it matches our expected size for X. What should that size be?
# Hint: https://pytorch.org/docs/stable/generated/torch.reshape.html

X = input_embedding_layer(input_token_ids).squeeze()
print(X.shape)

torch.Size([7, 768])


In [36]:
first_ten_of_third_token_embedding = X[2][:10]
print(f"First 10 values in the third token embedding (submit this on the website): \n{[round(el.item(), 3) for el in first_ten_of_third_token_embedding]}")

First 10 values in the third token embedding (submit this on the website): 
[-0.125, -0.064, 0.184, 0.009, 0.109, -0.126, -0.199, -0.191, -0.03, -0.09]


## 3.2.3 Query, Key, Value Vectors

In [38]:
# EDIT ME!

# Let's create our query, key, and value vectors.
# We have provided W_q, W_k, and W_v for you.
Q = X @ W_q
K = X @ W_k
V = X @ W_v

torch.Size([768, 768])
torch.Size([7, 768])


In [40]:
# EDIT ME!

# We want to inspect the *third* query, generated from the third embedding
# in the sequence. Again, we'll only look at the **first 10 values**.
first_ten_of_third_query = Q[2][:10]
print(f"First 10 values in the third query (submit this on the website): \n{[round(el.item(), 3) for el in first_ten_of_third_query]}")

first_ten_of_third_key = K[2][:10]
print(f"\n\nFirst 10 values in the third key (submit this on the website): \n{[round(el.item(), 3) for el in first_ten_of_third_key]}")

first_ten_of_third_value = V[2][:10]
print(f"\n\nFirst 10 values in the third value (submit this on the website): \n{[round(el.item(), 3) for el in first_ten_of_third_value]}")

First 10 values in the third query (submit this on the website): 
[0.19, 0.429, 0.272, 0.243, -0.12, -0.255, 0.619, 0.154, 0.012, 0.027]


First 10 values in the third key (submit this on the website): 
[-0.315, 0.163, -0.191, 0.102, 0.115, -0.238, -0.063, 0.231, 0.021, 0.078]


First 10 values in the third value (submit this on the website): 
[0.273, -0.246, 0.103, 0.562, 0.453, -0.511, -0.034, 0.165, -0.206, 0.217]


## 3.2.4 Attention and Output

In [41]:
# We have provided a function for rowwise softmax for you.
def softmax_rowwise(X):
    """Compute softmax values for each sets of scores in each row of matrix X."""
    e_x = torch.exp(X - torch.max(X, axis=1, keepdim=True).values)  # Subtract max per row for numerical stability
    return e_x / e_x.sum(axis=1, keepdim=True)

In [45]:
# EDIT ME!

# Calculate the attention matrix and the final output of the self-attention head
# below. Hint: Don't forget about the math.sqrt(d_k) in the attention calculation!
A = softmax_rowwise(Q @ K.T / math.sqrt(X.shape[1]))
output = A @ V
print(output.shape)

torch.Size([7, 768])


In [44]:
# EDIT ME!

first_ten_of_third_output = output[2][:10]
print(f"First 10 values in the third output vector (submit this on the website): \n{[round(el.item(), 3) for el in first_ten_of_third_output]}")

First 10 values in the third output vector (submit this on the website): 
[-0.904, -0.162, -0.3, -0.773, 0.368, 0.394, 0.382, 0.486, 0.221, -0.719]


## 3.2.5 Translating Back to Text

In [46]:
# Now that we have our output embeddings, let's see how we did.
# In order to turn our output back into text, we first need to translate our
# embeddings back into tokens.

# Get the output embedding layer (the final layer of GPT-2).
# This layer takes in a tensor of size (num_tokens, embedding_size)
# and returns a tensor of size (num_tokens, vocabulary_size) where GPT-2's
# vocabulary size is 50257.
output_embedding_layer = model.get_output_embeddings()

In [47]:
# EDIT ME!

# Send the calculated `output` through the `output_embedding_layer`.
# We'll call this `logits`.
logits = output_embedding_layer(output)

# We can interpret `logits` as the model's prediction of which token IDs are most
# likely to match each given embedding. For instance, consider the first row of
# `logits`: the first value in this row tells us how much the model thinks that this
# first output vector could match the token with ID = 0 (which is "!"). The second value in this row
# tells us how much the model thinks that the first output vector could match
# token ID = 1 (which is a quotation mark, """). And so on.
# Larger values indicate higher likelihood, so the index of the largest value can be
# considered the predicted output token ID.
# Get the predicted token ID for each output vector.
predicted_token_ids = logits.argmax(dim=1)

# Finally, we decode the token IDs back into text.
output_text = tokenizer.decode(predicted_token_ids, skip_special_tokens=False)

print(f"Output text (submit this on the website!): , '{output_text}'")
print("Output token IDs: ", predicted_token_ids.tolist())

Output text (submit this on the website!): , 'This is good    '
Output token IDs:  [1212, 318, 922, 220, 220, 220, 220]


# 4. Permutation attention

In [62]:
import numpy as np

X = np.array([[-0.83, 0.51, 0.49], [-0.41, 0.45, 0.06], [0.26, -0.13, -0.14], [-0.64, 0.41, 0.29], [0.39, 0.05, -0.19]])
Xnew = np.array([[-0.83,  0.51,  0.49], [-0.64,  0.41,  0.29],[-0.41,  0.45,  0.06],[ 0.26, -0.13, -0.14],  [ 0.39,  0.05, -0.19]])
Wq = np.array([[-0.44, -0.29, 0.77, 0.20], [-0.72, -0.22, -0.89, 0.36], [-0.63, -0.73, -0.46, 0.62]])
Wk = np.array([[0.86, -0.01, -0.75, -1.00], [0.49, 0.92, 0.60, 0.93], [-0.50, -0.30, 0.55, 0.09]])
Wv = np.array([[0.45, -0.36, -0.90, 0.88], [0.63, 0.54, 0.48, -1.00], [-0.84, 0.75, 0.81, 0.19]])

def softmax_rowwise(x):
    """Compute softmax values for each sets of scores in each row of matrix x."""
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # Subtract max per row for numerical stability
    return e_x / e_x.sum(axis=1, keepdims=True)

Q = X @ Wq
K = X @ Wk
V = X @ Wv

A = softmax_rowwise(Q @ K.T / 2)
output = A @ V


Q = Xnew @ Wq
K = Xnew @ Wk
V = Xnew @ Wv
A = softmax_rowwise(Q @ K.T / 2)
print(A)
output = A @ V
print(output.tolist())

[[0.14085632 0.16012385 0.17406324 0.26759704 0.25735956]
 [0.15019389 0.16695172 0.18019713 0.25394305 0.24871421]
 [0.16198002 0.17539002 0.18593331 0.24005248 0.23664418]
 [0.22069807 0.2118632  0.20570213 0.18014716 0.18158944]
 [0.23168768 0.21819209 0.20574952 0.17320603 0.17116468]]
[[0.03457854735745381, 0.1778488367956899, 0.23382495875156276, -0.29269743083612193], [0.02342547901205439, 0.19976638740433786, 0.26731188473602996, -0.3213411083054296], [0.009385703553449372, 0.22583432045651008, 0.3073138264000612, -0.3549136863731004], [-0.05618569442510728, 0.34434296586021507, 0.48916015808564317, -0.5056252499723483], [-0.06789168486975396, 0.363481923998439, 0.5186868226475315, -0.5290681197608346]]


# 5. Positional encoding

In [49]:
import numpy as np

def get_positional_encoding(n, d):
    """
    Generate sinusoidal positional encoding as used in "Attention Is All You Need"
    """
    position_enc = np.zeros((n, d))
    positions = np.arange(n)[:, np.newaxis]  # Shape: (n, 1)

    # Calculate how many even indices we have (for both odd and even d)
    even_indices = d // 2 + d % 2  # This works for both odd and even d
    div_term = (1/10000.0) ** (2*np.arange(0, even_indices)/d)

    # Apply sine to even indices (0, 2, 4, ...)
    position_enc[:, 0::2] = np.sin(positions * div_term[:d//2 + d%2])

    # Apply cosine to odd indices (1, 3, 5, ...)
    # Make sure we only use as many values as we have odd indices
    odd_indices = d // 2  # This works for both odd and even d
    position_enc[:, 1::2] = np.cos(positions * div_term[:odd_indices])

    return position_enc

In [66]:
# EDIT ME!
# 5.2 Applying Positional Encoding
# Recompute the attention output, but this time using positional encoding.
# X, Wq, Wk, Wv, and softmax_rowwise are defined in section 4 above.

# X_new is the input from problem 4.3 ("this very homework is fun"):
X_new = np.array([[-0.83,  0.51,  0.49], [-0.64,  0.41,  0.29],[-0.41,  0.45,  0.06],[ 0.26, -0.13, -0.14],  [ 0.39,  0.05, -0.19]])

# Generate positional encodings P with shape (n, d) and add to X_new.
n, d = X_new.shape
P = get_positional_encoding(n, d)
X_with_pos = X_new + P

# Compute Q, K, V from the position-enhanced embeddings.
Q = X_with_pos @ Wq
K = X_with_pos @ Wk
V = X_with_pos @ Wv

# Compute the attention matrix and the output Z_new.
# Hint: dont forget the sqrt(d_k) scaling inside softmax.
A = softmax_rowwise(Q @ K.T / math.sqrt(Q.shape[1]))
Z_new = A @ V

print("Z_new (submit this on the website!):")
print(Z_new.tolist())

print(get_positional_encoding(5, 4))

Z_new (submit this on the website!):
[[-0.21983455375324074, -0.37146782634957837, -0.41599851966001467, 0.6271829978185541], [-0.1168703408748218, -0.04713502804237074, -0.030169083872646555, 0.13114513870150743], [0.014278745201296243, 0.311080754510038, 0.36620198519276526, -0.37615638512256017], [0.12890903392269873, 0.5861642973690194, 0.6559560238684156, -0.7516138164045784], [0.06069854027478964, 0.23275608200302725, 0.2133524304125262, -0.2145922330664591]]
[[ 0.          1.          0.          1.        ]
 [ 0.84147098  0.54030231  0.00999983  0.99995   ]
 [ 0.90929743 -0.41614684  0.01999867  0.99980001]
 [ 0.14112001 -0.9899925   0.0299955   0.99955003]
 [-0.7568025  -0.65364362  0.03998933  0.99920011]]


# 6. [Optional] foodGPT

In [ ]:
import os
import torch
import traceback
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
# hyperparameters
# max_iters = 1000   # use this if training from scratch
max_iters = 100      # use this if loading saved state
eval_interval = 100
learning_rate = 3e-4
device = 'cpu'
eval_iters = 200
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?

d = 64
n_head = 4
n_layer = 4

In [ ]:
# load input data
!wget --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw09/food.txt
!wget --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw09/foodGPT_state.pth

sfn = "foodGPT_state.pth"
torch.manual_seed(1337)

with open('food.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    text = text.lower()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
# train_data = data[:n]
train_data = data
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# GPT model

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, d_k):
        super().__init__()
        self.key = nn.Linear(d, d_k, bias=False)
        self.query = nn.Linear(d, d_k, bias=False)
        self.value = nn.Linear(d, d_k, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, d_k):
        super().__init__()
        self.heads = nn.ModuleList([Head(d_k) for _ in range(num_heads)])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.ReLU(),
            nn.Linear(4 * d, d),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, d, n_head):
        # d: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        d_k = d // n_head
        self.sa = MultiHeadAttention(n_head, d_k)
        self.ffwd = FeedFoward(d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, d)
        self.position_embedding_table = nn.Embedding(block_size, d)
        self.blocks = nn.Sequential(*[Block(d, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d) # final layer norm
        self.lm_head = nn.Linear(d, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, end_token=None):
        '''
        idx is (B, T) array of indices in the current context
        If end_token is provided, then terminate when that is generated
        '''
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            if end_token is not None and idx_next==end_token:
                break
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [ ]:
# train model

model = GPTLanguageModel()

if os.path.exists(sfn):
    try:
        mstate = torch.load(sfn)
        print(f"Read model state from {sfn}")
        model.load_state_dict(mstate)
    except Exception as err:
        print("Oops, failed to load saved state, err={err}")

m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# save model state
torch.save(m.state_dict(), sfn)
print(f"Wrote model state to {sfn}")

In [ ]:
#-----------------------------------------------------------------------------
# generate output from the model
# accept input from user and print results from GPT
# stop on empty input

while True:
    args = input("Enter some characters as a prompt: ")
    try:
        prompt = args.strip().lower()
        if not prompt:
            break
        context = torch.tensor([ encode(prompt) ], dtype=torch.long, device=device)
        ret = m.generate(context, max_new_tokens=100, end_token=0)[0].tolist()
        # print("ret=", ret)
        ret = decode(ret)
        print(ret)
    except Exception as err:
        print(traceback.format_exc())
        print(err)

In [ ]:
# generate one character at a time.  Show the top five outputs, and prompt
# user to select one, then repeat.
# terminate on empty input

def get_next(prompt):
    context = torch.tensor([ encode(prompt) ], dtype=torch.long, device=device)
    logits, loss = m(context)    # evaluate model on input, obtaining softmax output and loss
    logits = logits[:, -1, :]    # get output for the very last character
    # EDIT: Your code here
    idx_next = idx_next[0].tolist()
    print(idx_next)
    cnext = decode(idx_next)
    return cnext

while True:
    prompt = input("Enter some characters as a prompt: ").strip().lower()
    if not prompt:
        break
    try:
        while True:
            cnext = get_next(prompt).strip()
            print(f"    Top 5 candidate next characters: {cnext}")
            achar = input(f"    prompt='{prompt}' ; enter next character: ").strip()
            if not achar:
                break
            prompt = prompt + achar
    except Exception as err:
        print(traceback.format_exc())
        print(err)

# 7. A Tale of Two Architectures: ViTs vs. CNNs

In this notebook, we'll explore *how* a model's focus changes depending on *what we ask it to find*. We'll use the classic **Grad-CAM** technique to create heatmaps.

**Our Goal:** Compare the heatmaps for a **CNN (ResNet-50)** and a **ViT (Tiny ViT)** when we ask them to find the "Dog" vs. the "Cat" in the same image.

There is no code for you to write. Your task is to run the cells, observe the results, and read the discussion points.

## 7.1 Installs and Imports
We need `pytorch-grad-cam` for the main algorithm and `timm` (PyTorch Image Models) for a lightweight ViT model.

In [ ]:
!pip install grad-cam opencv-python-headless requests timm

In [ ]:
import cv2
import numpy as np
import torch
from torchvision import models
import timm # We'll use this for a tiny ViT model
import matplotlib.pyplot as plt
import requests
from PIL import Image
import gc # For garbage collection
from warnings import filterwarnings

# Import the main Grad-CAM algorithm
from pytorch_grad_cam import GradCAM

# Import utilities for preprocessing and visualizing
from pytorch_grad_cam.utils.image import (
    show_cam_on_image, preprocess_image
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Suppress warnings
filterwarnings('ignore')
print("All packages imported.")

## 7.2 Load Pre-Trained Models

* **CNN:** `resnet50`
* **ViT:** `vit_tiny_patch16_224` (a fast, lightweight ViT)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load CNN model
cnn_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).eval().to(device)

# Load a TINY ViT model from TIMM
vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=1000).eval().to(device)

print("Pre-trained models loaded and in evaluation mode.")

## 7.3 Load and Preprocess the Example Image

We'll use the "cat and dog" image. Both models are trained on **224x224** images, so we must resize our input to this exact size.

In [ ]:
image_url = "https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/both.png"
image_path = "both.png"
with open(image_path, 'wb') as f:
    f.write(requests.get(image_url).content)

# Load the image with OpenCV and convert to RGB
img_bgr = cv2.imread(image_path, 1)
img_rgb = img_bgr[:, :, ::-1]

# --- CRITICAL STEP: Resize to 224x224 ---
img_rgb_resized = cv2.resize(img_rgb, (224, 224))

# Normalize the image to [0, 1] for visualization
img_float_norm = np.float32(img_rgb_resized) / 255

# Preprocess the image for the models
input_tensor = preprocess_image(img_float_norm,
                                  mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225]).to(device)

# Show the original (resized) image
print("Original Image (resized to 224x224):")
plt.imshow(img_float_norm)
plt.axis('off')
plt.show()

## 7.4 Helper Function for ViT

We still need our `reshape_transform` to convert the ViT's 1D token sequence back into a 2D grid for visualization. This function remains the same.

In [ ]:
def vit_reshape_transform(tensor, height=14, width=14):
    """
    Transforms the 1D token sequence from a ViT back into a 2D grid.
    (B, num_tokens, embed_dim) -> (B, embed_dim, H, W)
    """
    result = tensor[:, 1:, :] # Remove [CLS] token
    result = result.reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.permute(0, 3, 1, 2)
    return result

## 7.5 Targeted CAM Comparison

This is the main experiment. Instead of hard-coding class IDs, we'll **peek at the current model predictions** and automatically pick the highest-scoring "dog" and "cat" classes. On the provided image this resolves to:

* **Dog target:** Class `243` (*bull mastiff*)
* **Cat target:** Class `281` (*tabby, tabby cat*)

Your exact numbers may differ if you swap in a new image or fine-tune the models.

In [ ]:
try:
    # 1. Define target layers and class
    dog_target_idx = 243
    target_layers_cnn = [cnn_model.layer4]
    target_layers_vit = [vit_model.blocks[-1].norm1]
    targets = [ClassifierOutputTarget(dog_target_idx)]

    # 2. Clear gradients
    cnn_model.zero_grad()
    vit_model.zero_grad()
    input_tensor.grad = None

    # 3. Run Grad-CAM for CNN
    with GradCAM(model=cnn_model, target_layers=target_layers_cnn) as cam:
        input_tensor.requires_grad_(True)
        grayscale_cam_cnn = cam(input_tensor=input_tensor, targets=targets)[0, :]
        input_tensor.grad = None
    viz_cnn = show_cam_on_image(img_float_norm, grayscale_cam_cnn, use_rgb=True)

    # 4. Run Grad-CAM for ViT
    with GradCAM(model=vit_model,
                 target_layers=target_layers_vit,
                 reshape_transform=vit_reshape_transform) as cam:
        input_tensor.requires_grad_(True)
        grayscale_cam_vit = cam(input_tensor=input_tensor, targets=targets)[0, :]
        input_tensor.grad = None
    viz_vit = show_cam_on_image(img_float_norm, grayscale_cam_vit, use_rgb=True)

    # 5. Plot the results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f'Target: "Dog"', fontsize=16)
    ax1.imshow(viz_cnn)
    ax1.set_title("CNN: ResNet-50")
    ax1.axis('off')
    ax2.imshow(viz_vit)
    ax2.set_title("ViT: Tiny")
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

    # 6. --- MEMORY CLEANUP (inside try) ---
    del grayscale_cam_cnn, grayscale_cam_vit, viz_cnn, viz_vit

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # 7. --- CUDA CLEANUP (always run) ---
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
try:
    # 1. Define target layers and class
    cat_target_idx = 281
    target_layers_cnn = [cnn_model.layer4]
    target_layers_vit = [vit_model.blocks[-1].norm1]
    targets = [ClassifierOutputTarget(cat_target_idx)]

    # 2. Clear gradients
    cnn_model.zero_grad()
    vit_model.zero_grad()
    input_tensor.grad = None

    # 3. Run Grad-CAM for CNN
    with GradCAM(model=cnn_model, target_layers=target_layers_cnn) as cam:
        input_tensor.requires_grad_(True)
        grayscale_cam_cnn = cam(input_tensor=input_tensor, targets=targets)[0, :]
        input_tensor.grad = None
    viz_cnn = show_cam_on_image(img_float_norm, grayscale_cam_cnn, use_rgb=True)

    # 4. Run Grad-CAM for ViT
    with GradCAM(model=vit_model,
                 target_layers=target_layers_vit,
                 reshape_transform=vit_reshape_transform) as cam:
        input_tensor.requires_grad_(True)
        grayscale_cam_vit = cam(input_tensor=input_tensor, targets=targets)[0, :]
        input_tensor.grad = None
    viz_vit = show_cam_on_image(img_float_norm, grayscale_cam_vit, use_rgb=True)

    # 5. Plot the results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f'Target: "Cat"', fontsize=16)
    ax1.imshow(viz_cnn)
    ax1.set_title("CNN: ResNet-50")
    ax1.axis('off')
    ax2.imshow(viz_vit)
    ax2.set_title("ViT: Tiny")
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

    # 6. --- MEMORY CLEANUP (inside try) ---
    del grayscale_cam_cnn, grayscale_cam_vit, viz_cnn, viz_vit

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # 7. --- CUDA CLEANUP (always run) ---
    gc.collect()
    torch.cuda.empty_cache()

## 7.6 Try Your Own Image!

Run the cell below to test other images. **Just uncomment one of the `image_url` lines** (or paste your own).

This will run Grad-CAM for the **top-predicted class** (`targets=None`). You can also try setting a specific class index from the lookup tool!

In [ ]:
# --- CHOOSE YOUR IMAGE ---
# (Uncomment one line to select it)

image_url = "https://images.pexels.com/photos/170811/pexels-photo-170811.jpeg" # Car
# image_url = "https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg" # Two dogs
# image_url = "YOUR_OWN_IMAGE_URL.jpg" # Paste your own URL here

# --- CHOOSE YOUR TARGET ---
# targets = None # Top-predicted class
# targets = [ClassifierOutputTarget(YOUR_CLASS_INDEX)] # e.g., 87 for 'kingfisher'
targets = None # Default to top-predicted class

# ---------------------------------

print(f"Loading custom image from: {image_url}")

try:
    # 1. Load and preprocess image
    img_pil = Image.open(requests.get(image_url, stream=True).raw).convert('RGB')
    img_rgb_custom = np.array(img_pil)
    img_rgb_custom_resized = cv2.resize(img_rgb_custom, (224, 224))
    img_float_norm_custom = np.float32(img_rgb_custom_resized) / 255
    input_tensor_custom = preprocess_image(img_float_norm_custom,
                                         mean=[0.485, 0.456, 0.406],
                                         std=[0.229, 0.224, 0.225]).to(device)

    # 2. Define models to run
    models_to_run = [
        {"name": "CNN: ResNet-50",
         "model": cnn_model,
         "target_layers": [cnn_model.layer4],
         "transform": None},
        {"name": "ViT: Tiny",
         "model": vit_model,
         "target_layers": [vit_model.blocks[-1].norm1],
         "transform": vit_reshape_transform}
    ]

    # 3. Plot results
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_float_norm_custom)
    axes[0].set_title("Your Custom Image")
    axes[0].axis('off')

    viz_results = [] # To store viz for deletion

    for i, model_info in enumerate(models_to_run):
        ax = axes[i+1]
        ax.set_title(f"{model_info['name']}")
        ax.axis('off')

        try:
            model_info['model'].zero_grad()
            input_tensor_custom.grad = None

            with GradCAM(model=model_info['model'],
                         target_layers=model_info['target_layers'],
                         reshape_transform=model_info['transform']) as cam:

                input_tensor_custom.requires_grad_(True)
                grayscale_cam = cam(input_tensor=input_tensor_custom, targets=targets)[0, :]
                input_tensor_custom.grad = None

            viz = show_cam_on_image(img_float_norm_custom, grayscale_cam, use_rgb=True)
            viz_results.append(viz) # Save for deletion
            viz_results.append(grayscale_cam) # Save for deletion
            ax.imshow(viz)

        except Exception as e:
            print(f"    Failed to run {model_info['name']}. Error: {e}")
            ax.text(0.5, 0.5, "Error", ha='center', va='center')

    if targets is None:
        fig.suptitle("Target: Top Predicted Class", fontsize=16)
    else:
        try:
            fig.suptitle(f"Target: ({targets[0].target})", fontsize=16)
        except:
             fig.suptitle(f"Target: Class {targets[0].target}", fontsize=16)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # 4. --- MEMORY CLEANUP (inside try) ---
    del input_tensor_custom
    for item in viz_results:
        del item

except Exception as e:
    print(f"An error occurred. Is the URL correct and a direct link to an image? Error: {e}")

finally:
    # 5. --- CUDA CLEANUP (always run) ---
    gc.collect()
    torch.cuda.empty_cache()

# [Optional] Sentence Transformer and food for thought

In [ ]:
!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# example of using SentenceTransformer
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Sentences we want to encode. Example:
sentence = ['This framework generates embeddings for each input sentence']

# Sentences are encoded by calling model.encode()
embedding = model.encode(sentence)
print(f"The sentence '{sentence}' embedded as a vector of length {len(embedding[0])} is {embedding[0]}")

In [ ]:
!wget --no-check-certificate https://introml.mit.edu/_static/spring26/homework/hw09/food.txt
food_vectors = []
foods = []
cnt = 0
for line in open("food.txt"):
    line = line.strip()
    if (cnt < 10):
        print(f"{cnt+1}: {line}")
    cnt += 1
    foods.append(line)
    food_vectors.append(model.encode(line))
print(f"--> Encoded {len(food_vectors)} food vectors")
food_matrix = np.array(food_vectors)
print(f"food_matrix shape={food_matrix.shape}")  # each row is one food

In [ ]:
while True:
    x = input("Enter phrase to match: ").strip()
    if not x:
        break
    xvec = np.array(model.encode(x))
    # your code here, to find the closest match
    print("Top 5 matches are:")
    print([foods[i] for i in top5])